# Analysis & Visual Outputs

This notebook covers the **visual outputs** part of the internship instructions:
plots, metrics, and a confusion matrix.

It analyses two things:
1. The original customer-service FAQ dataset (insights + plots).
2. The Task 5 sentiment analyzer (accuracy + confusion matrix on a small labelled set).

Run cells top to bottom.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. FAQ dataset insights

In [ ]:
# Load the FAQ dataset used by the base chatbot.
df = pd.read_csv('dataset/dataset.csv')
print('Number of FAQ pairs:', len(df))
df.head()

In [ ]:
# Feature engineering: measure how long each question and answer is (word count).
df['question_words'] = df['prompt'].astype(str).str.split().str.len()
df['answer_words'] = df['response'].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['question_words'], bins=20, ax=axes[0], color='steelblue')
axes[0].set_title('Question length (words)')
sns.histplot(df['answer_words'], bins=20, ax=axes[1], color='seagreen')
axes[1].set_title('Answer length (words)')
plt.tight_layout()
plt.show()

In [ ]:
# Insight: most common starting words in questions (a simple intent hint).
first_word = df['prompt'].astype(str).str.split().str[0].str.lower()
top = first_word.value_counts().head(10)

plt.figure(figsize=(8, 4))
sns.barplot(x=top.values, y=top.index, color='slateblue')
plt.title('Top 10 question starting words')
plt.xlabel('Count')
plt.show()

## 2. Sentiment analyzer evaluation (Task 5)

We test the VADER-based analyzer on a small hand-labelled set and show a
confusion matrix. This is the 'metrics + confusion matrix' deliverable.

In [ ]:
import sys
sys.path.append('src')  # task modules live in the src folder
from tasks.task5_sentiment import detect_sentiment

# Small labelled test set: (message, true_label)
test_data = [
    ('This is the worst service ever, I am so angry!', 'negative'),
    ('You people are useless, I want a refund now.', 'negative'),
    ('Terrible experience, never buying again.', 'negative'),
    ('Thank you so much, you have been wonderful!', 'positive'),
    ('I love this course, it is amazing.', 'positive'),
    ('Great support, very happy with the help.', 'positive'),
    ('What are your office hours?', 'neutral'),
    ('Please tell me the refund policy.', 'neutral'),
    ('I would like to update my address.', 'neutral'),
]

true_labels = [t[1] for t in test_data]
pred_labels = [detect_sentiment(t[0]) for t in test_data]
list(zip([t[0] for t in test_data], true_labels, pred_labels))

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

labels = ['negative', 'neutral', 'positive']
cm = confusion_matrix(true_labels, pred_labels, labels=labels)

print('Accuracy:', round(accuracy_score(true_labels, pred_labels), 2))
print()
print(classification_report(true_labels, pred_labels, labels=labels, zero_division=0))

In [ ]:
# Confusion matrix heatmap.
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Sentiment confusion matrix (VADER)')
plt.tight_layout()
plt.show()

### Notes / honest limitations
- VADER is rule-based, so neutral business questions containing mild positive
  words (like 'please' or 'want') can be mislabelled positive. The confusion
  matrix above usually shows this in the neutral row.
- This is acceptable for a fast, offline demo. For higher accuracy you could
  replace `detect_sentiment` with a Gemini-based classifier or a fine-tuned
  model, then re-run this same notebook to compare (baseline vs advanced).